# DRSAC — Geração de Arquivo XML Regulatório

Este notebook demonstra o pipeline de geração do **DRSAC (Documento de Risco Socioambiental e Climático)**, relatório regulatório exigido pelo Banco Central do Brasil (BACEN) para instituições financeiras.

## O que este código faz
1. **Lê uma planilha** com dados de clientes e operações de crédito
2. **Agrupa** operações por cliente, evitando duplicatas
3. **Gera um XML** hierárquico com avaliações de risco (social, ambiental e climático) por operação

## Estrutura do XML
```
DocumentoDRSAC
  ├── Contato
  └── Clientes
        └── Cliente
              ├── ExpAtivos
              │     └── ExpOperCred (por operação)
              │           ├── RiscSoc, RiscAmb, RiscClimFis, RiscClimTrans
              │           ├── ContribPositiva, MitRiscClimFis
              │           ├── HistAbsorEmissGEE, CompEmissGEE
              │           └── LocalizCEP
              └── ExpCliente
                    └── RiscSoc, RiscAmb, RiscClimFis, RiscClimTrans, ...
```

> ⚠️ **Nota:** Os dados neste notebook são **fictícios**, gerados para fins de demonstração.

## 1. Dependências

In [ ]:
# pip install pandas openpyxl

import pandas as pd
from xml.etree.ElementTree import Element, SubElement
from io import StringIO

## 2. Funções Auxiliares

In [ ]:
def as_text(v):
    """
    Converte um valor para string segura.
    Remove o sufixo '.0' gerado quando o Excel lê inteiros como float.
    Retorna string vazia para valores nulos.
    """
    if pd.isna(v):
        return ""
    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


def format_element_with_newlines(elem, level=0):
    """Serializa um elemento XML com indentação legível."""
    indent = "    " * level
    attributes = "".join([f'\n{indent}    {k}="{v}"' for k, v in elem.attrib.items()])
    if list(elem):
        content = "".join([format_element_with_newlines(child, level + 1) for child in elem])
        return f"{indent}<{elem.tag}{attributes}>\n{content}{indent}</{elem.tag}>\n"
    else:
        return f"{indent}<{elem.tag}{attributes}/>\n"


print("Funções auxiliares definidas.")

## 3. Carregamento dos Dados

Em produção, substitua o DataFrame fictício abaixo pela leitura real da planilha:
```python
df = pd.read_excel("caminho/para/Dados_DRSAC.xlsx", engine="openpyxl")
```

### Colunas esperadas na planilha

| Coluna | Descrição |
|---|---|
| `cnpj` | CNPJ do cliente (sem os 2 primeiros dígitos) |
| `dataBase` | Competência no formato YYYY-MM |
| `codigoDocumento` | Código do documento BACEN |
| `tipoEnvio` | Tipo de envio (ex: `N` = normal) |
| `nome`, `fone`, `email` | Dados do responsável pelo envio |
| `ident` | Identificador único do cliente |
| `CNAE`, `versaoCNAE` | Classificação de atividade econômica |
| `IPOC`, `Sicor` | Identificadores da operação de crédito |
| `saldo` | Saldo da operação em reais |
| `CEP` | CEP da operação |
| `RiscSoc_01..99` | Avaliação de risco social por tipo |
| `RiscAmb_01..99` | Avaliação de risco ambiental por tipo |
| `RiscClimFis_01..99` | Risco climático físico por tipo |
| `RiscClimTrans_01..99` | Risco climático de transição por tipo |

In [ ]:
# Dados fictícios representando a estrutura esperada da planilha
mock_csv = """
cnpj,dataBase,codigoDocumento,tipoEnvio,nome,fone,email,ident,tipo,CNAE,versaoCNAE,IPOC,Sicor,saldo,CEP,RiscSoc_01,RiscSoc_02,RiscSoc_03,RiscSoc_04,RiscSoc_05,RiscSoc_99,RiscAmb_01,RiscAmb_02,RiscAmb_03,RiscAmb_04,RiscAmb_05,RiscAmb_06,RiscAmb_07,RiscAmb_08,RiscAmb_09,RiscAmb_99,RiscClimFis_01,RiscClimFis_02,RiscClimFis_03,RiscClimFis_99,RiscClimTrans_01,RiscClimTrans_02,RiscClimTrans_03,RiscClimTrans_04,RiscClimTrans_99
12345678000100,2025-12,2030,N,Nome Responsavel,1100000000,responsavel@instituicao.com.br,12345678000100,02,0111300,2.0,IPOC001,,50000.00,01310100,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
12345678000100,2025-12,2030,N,Nome Responsavel,1100000000,responsavel@instituicao.com.br,12345678000100,02,0111300,2.0,IPOC002,,30000.00,01310100,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
98765432000155,2025-12,2030,N,Nome Responsavel,1100000000,responsavel@instituicao.com.br,98765432000155,02,0121101,2.0,IPOC003,SICOR001,120000.00,04538133,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1
""".strip()

df = pd.read_csv(StringIO(mock_csv))
print(f"DataFrame carregado: {len(df)} linhas, {len(df.columns)} colunas")
df.head()

## 4. Geração do XML DRSAC

In [ ]:
# ── Constantes de risco (conforme manual BACEN) ──────────────────────────────
TIPOS_RISCSOC       = ['01', '02', '03', '04', '05', '99']
TIPOS_RISCAMB       = [f'{i:02d}' for i in range(1, 10)] + ['99']
TIPOS_RISCCLIMFIS   = ['01', '02', '03', '99']
TIPOS_RISCCLIMTRANS = ['01', '02', '03', '04', '99']

# ── Cabeçalho do documento ───────────────────────────────────────────────────
first = df.iloc[0]

documento = Element('DocumentoDRSAC', {
    'cnpj': "00" + as_text(first['cnpj']),   # BACEN exige prefixo '00'
    'dataBase': as_text(first['dataBase']),
    'codigoDocumento': as_text(first['codigoDocumento']),
    'tipoEnvio': as_text(first['tipoEnvio'])
})

SubElement(documento, 'Contato', {
    'nome': as_text(first['nome']),
    'fone': as_text(first['fone']),
    'email': as_text(first['email'])
})

clientes_elem = SubElement(documento, 'Clientes')

# Dicionário para evitar duplicar o nó <Cliente> quando há múltiplas operações
clientes_dict = {}

# ── Iteração por linha (cada linha = uma operação de crédito) ─────────────────
for _, row in df.iterrows():
    if pd.isna(row.get('ident')):
        continue

    ident = as_text(row['ident'])

    # ── Cria o nó <Cliente> apenas uma vez por cliente ────────────────────────
    if ident not in clientes_dict:
        cliente = SubElement(clientes_elem, 'Cliente', {
            'ident': ident,
            'tipo': "02",
            'CNAE': as_text(row['CNAE']),
            'versaoCNAE': as_text(row['versaoCNAE'])
        })

        exp_ativos  = SubElement(cliente, 'ExpAtivos')
        exp_cliente = SubElement(cliente, 'ExpCliente')

        # Avaliações de risco consolidadas do cliente
        for tipo in TIPOS_RISCSOC:
            SubElement(exp_cliente, 'RiscSoc',       {'tipo': tipo, 'av': "0" + as_text(row.get(f'RiscSoc_{tipo}', 0))})
        for tipo in TIPOS_RISCAMB:
            SubElement(exp_cliente, 'RiscAmb',       {'tipo': tipo, 'av': "0" + as_text(row.get(f'RiscAmb_{tipo}', 0))})
        for tipo in TIPOS_RISCCLIMFIS:
            SubElement(exp_cliente, 'RiscClimFis',   {'tipo': tipo, 'av': "0" + as_text(row.get(f'RiscClimFis_{tipo}', 0))})
        for tipo in TIPOS_RISCCLIMTRANS:
            SubElement(exp_cliente, 'RiscClimTrans', {'tipo': tipo, 'av': as_text(row.get(f'RiscClimTrans_{tipo}', 0))})

        # Subelementos obrigatórios com valores padrão
        for enquad in ['01', '02', '03', '98', '99']:
            SubElement(exp_cliente, 'DetContribPositiva', {'enquad': enquad, 'saldoCred': '0.00', 'saldoTVM': '0.00'})
        for tipo in ['01', '02', '03', '04']:
            SubElement(exp_cliente, 'HistAbsorEmissGEE', {'tipo': tipo, 'sit': '99', 'valor': '0.00'})
        for tipo in ['01', '02', '03', '04']:
            SubElement(exp_cliente, 'ExpAbsorEmissGEE',  {'tipo': tipo, 'sit': '99', 'valor': '0.00'})
        for tipo in ['01', '02']:
            SubElement(exp_cliente, 'CompEmissGEE',      {'tipo': tipo, 'sit': '99', 'valor': '0.00'})
        for tipo in [f'{i:02d}' for i in range(1, 11)]:
            SubElement(exp_cliente, 'AgrMit', {'tipo': tipo, 'sit': '99'})

        clientes_dict[ident] = (cliente, exp_ativos, exp_cliente)

    # ── Cria uma <ExpOperCred> para CADA linha (cada operação) ────────────────
    cliente, exp_ativos, exp_cliente = clientes_dict[ident]

    exp_opercred = SubElement(exp_ativos, 'ExpOperCred', {
        'IPOC':  as_text(row['IPOC']),
        'Sicor': as_text(row['Sicor']),
        'saldo': f"{float(row['saldo']):.2f}" if not pd.isna(row['saldo']) else "0.00"
    })

    # Avaliações de risco por operação
    for tipo in TIPOS_RISCSOC:
        SubElement(exp_opercred, 'RiscSoc',       {'tipo': tipo, 'av': "0" + as_text(row.get(f'RiscSoc_{tipo}', 0))})
    for tipo in TIPOS_RISCAMB:
        SubElement(exp_opercred, 'RiscAmb',       {'tipo': tipo, 'av': "0" + as_text(row.get(f'RiscAmb_{tipo}', 0))})
    for tipo in TIPOS_RISCCLIMFIS:
        SubElement(exp_opercred, 'RiscClimFis',   {'tipo': tipo, 'av': "0" + as_text(row.get(f'RiscClimFis_{tipo}', 0))})
    for tipo in TIPOS_RISCCLIMTRANS:
        SubElement(exp_opercred, 'RiscClimTrans', {'tipo': tipo, 'av': as_text(row.get(f'RiscClimTrans_{tipo}', 0))})

    SubElement(exp_opercred, 'ContribPositiva', {'enquad': '99'})
    SubElement(exp_opercred, 'MitRiscClimFis',  {'exist': '99'})

    for tipo in ['01', '02', '03', '04']:
        SubElement(exp_opercred, 'HistAbsorEmissGEE', {'tipo': tipo, 'sit': '99', 'valor': '0.00'})
    for tipo in ['01', '02']:
        SubElement(exp_opercred, 'CompEmissGEE',      {'tipo': tipo, 'sit': '99', 'valor': '0.00'})

    SubElement(exp_opercred, 'LocalizCEP', {'CEP': as_text(row['CEP'])})


print(f"✅ XML gerado com {len(clientes_dict)} clientes e {len(df)} operações.")

## 5. Exportação do Arquivo XML

In [ ]:
import os
from datetime import datetime

# Parâmetros de saída
COD_DOCUMENTO = as_text(first['codigoDocumento'])  # ex: 2030
DATA_BASE     = as_text(first['dataBase']).replace("-", "_")  # ex: 2025_12
OUTPUT_DIR    = "./output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Padrão de nome: <CodDoc>_<Ano>_<Mes>_<Remessa>.xml
nome_arquivo = f"{OUTPUT_DIR}/{COD_DOCUMENTO}_{DATA_BASE}_01.xml"

xml_str = '<?xml version="1.0" encoding="UTF-8"?>\n' + format_element_with_newlines(documento)
with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write(xml_str)

print(f"Arquivo gerado: {nome_arquivo}")
print("\nPrimeiras linhas do XML:")
print("\n".join(xml_str.split("\n")[:25]))